<a href="https://colab.research.google.com/github/kedwinparga21-wq/Proyecto-integrado/blob/main/EA1_carga_creacion_dataset_(1).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# EA1 – Carga y creación de dataset
**Proyecto integrador – Analítica de Datos** · Dataset: `infinite-dataset-hub/RetailSalesPerformance` (Hugging Face)

Este notebook coordina el flujo pedido en la actividad:

`HFDatasetLoader (carga) → DataFrame → transformaciones → DatasetSummarizer (resumen) → archivo .txt`

- **Clase 1** (`hf_dataset_loader.py`): obtiene el dataset de Hugging Face y lo devuelve como DataFrame de pandas.
- **Clase 2** (`dataset_summarizer.py`): recibe un DataFrame y genera un resumen en un `.txt`.
- **Este notebook** solo coordina: importa las clases, aplica las transformaciones y muestra los resultados.

In [14]:
# Importar las dos clases desde sus archivos .py.
# Si el notebook corre en Colab y los archivos no están, se descargan del repositorio.
import urllib.request
from pathlib import Path

BASE_REPO = "https://raw.githubusercontent.com/kedwinparga21-wq/Proyecto-integrado/main/"
for archivo in ("hf_dataset_loader.py", "dataset_summarizer.py"):
    if not Path(archivo).exists():
        urllib.request.urlretrieve(BASE_REPO + archivo, archivo)

import pandas as pd
from hf_dataset_loader import HFDatasetLoader
from dataset_summarizer import DatasetSummarizer

pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 160)
print("Clases importadas: HFDatasetLoader y DatasetSummarizer")

Clases importadas: HFDatasetLoader y DatasetSummarizer


## 1. Obtener el DataFrame con la primera clase
La conexión con Hugging Face y la carga de datos ocurren dentro de la clase.

In [15]:
cargador = HFDatasetLoader("infinite-dataset-hub/RetailSalesPerformance", split="train")
df_original = cargador.load()

print("Origen:", cargador.describe_source())
print("Filas y columnas:", df_original.shape)
df_original.head()

Origen: {'dataset_id': 'infinite-dataset-hub/RetailSalesPerformance', 'url': 'https://huggingface.co/datasets/infinite-dataset-hub/RetailSalesPerformance', 'split': 'train', 'config_name': None, 'revision': None}
Filas y columnas: (100, 10)


,idx,Date,OutletName,Location,ProductCategory,UnitsSold,Revenue,CustomerAgeGroup,CustomerIncomeLevel,Label
0,0,2023-01-01,Green Market,Springfield,Electronics,150,45000,30-45,High,High
1,1,2023-01-02,City Mall,Springfield,Home & Garden,200,50000,25-35,Medium,Medium
2,2,2023-01-03,Book Haven,Lakeview,Books,120,18000,18-24,Low,Low
3,3,2023-01-04,FashionFiesta,River City,Apparel,250,75000,40-50,High,High
4,4,2023-01-05,Food World,Springfield,Food & Beverages,300,60000,20-30,Medium,Medium


## 2. Transformaciones del DataFrame
Se aplican **seis transformaciones distintas**, cada una con su resultado visible. El DataFrame original se conserva sin cambios para poder comparar.

In [16]:
df = df_original.copy()
registro = []   # guarda qué se hizo en cada paso para dejarlo en el resumen .txt

**Transformación 1 – Renombrar columnas** a nombres claros en español y en formato `snake_case`.

In [17]:
nombres = {
    "idx": "id_registro", "Date": "fecha", "OutletName": "tienda", "Location": "ubicacion",
    "ProductCategory": "categoria_producto", "UnitsSold": "unidades_vendidas", "Revenue": "ingresos",
    "CustomerAgeGroup": "grupo_edad_cliente", "CustomerIncomeLevel": "nivel_ingreso_cliente", "Label": "etiqueta_desempeno",
}
df = df.rename(columns=nombres)
registro.append("T1: se renombraron las 10 columnas a español en formato snake_case.")
print(list(df.columns))

['id_registro', 'fecha', 'tienda', 'ubicacion', 'categoria_producto', 'unidades_vendidas', 'ingresos', 'grupo_edad_cliente', 'nivel_ingreso_cliente', 'etiqueta_desempeno']


**Transformación 2 – Convertir tipos de datos:** la fecha llega como texto y se convierte a fecha.

In [18]:
print("Tipo de fecha antes:", df["fecha"].dtype)
df["fecha"] = pd.to_datetime(df["fecha"], errors="coerce")
print("Tipo de fecha después:", df["fecha"].dtype, "| fechas inválidas:", int(df["fecha"].isna().sum()))
registro.append("T2: la columna fecha se convirtió de texto a datetime.")

Tipo de fecha antes: object
Tipo de fecha después: datetime64[ns] | fechas inválidas: 0


**Transformación 3 – Normalizar texto:** se quitan espacios sobrantes en las columnas de texto para evitar categorías duplicadas.

In [19]:
columnas_texto = ["tienda", "ubicacion", "categoria_producto", "grupo_edad_cliente", "nivel_ingreso_cliente", "etiqueta_desempeno"]
for columna in columnas_texto:
    df[columna] = df[columna].astype("string").str.strip().str.replace(r"\s+", " ", regex=True)
registro.append("T3: se normalizaron espacios en las 6 columnas de texto.")
df[columnas_texto].head()

,tienda,ubicacion,categoria_producto,grupo_edad_cliente,nivel_ingreso_cliente,etiqueta_desempeno
0,Green Market,Springfield,Electronics,30-45,High,High
1,City Mall,Springfield,Home & Garden,25-35,Medium,Medium
2,Book Haven,Lakeview,Books,18-24,Low,Low
3,FashionFiesta,River City,Apparel,40-50,High,High
4,Food World,Springfield,Food & Beverages,20-30,Medium,Medium


**Transformación 4 – Crear columnas calculadas:** precio por unidad y mes de la venta.

In [20]:
df["precio_unitario"] = (df["ingresos"] / df["unidades_vendidas"]).round(2)
df["mes"] = df["fecha"].dt.strftime("%Y-%m")
registro.append("T4: se crearon precio_unitario (ingresos / unidades_vendidas) y mes (AAAA-MM).")
df[["fecha", "mes", "unidades_vendidas", "ingresos", "precio_unitario"]].head()

,fecha,mes,unidades_vendidas,ingresos,precio_unitario
0,2023-01-01,2023-01,150,45000,300.0
1,2023-01-02,2023-01,200,50000,250.0
2,2023-01-03,2023-01,120,18000,150.0
3,2023-01-04,2023-01,250,75000,300.0
4,2023-01-05,2023-01,300,60000,200.0


**Transformación 5 – Seleccionar columnas pertinentes:** se descarta `id_registro` porque es solo un consecutivo y no aporta al análisis.

In [21]:
df = df.drop(columns=["id_registro"])
registro.append("T5: se eliminó id_registro (consecutivo sin valor analítico).")
print("Columnas actuales:", list(df.columns))

Columnas actuales: ['fecha', 'tienda', 'ubicacion', 'categoria_producto', 'unidades_vendidas', 'ingresos', 'grupo_edad_cliente', 'nivel_ingreso_cliente', 'etiqueta_desempeno', 'precio_unitario', 'mes']


**Transformación 6 – Filtrar filas con una condición:** se conservan las ventas de **100 unidades o más**, para quedarse con las ventas de mayor volumen.

In [22]:
filas_antes = len(df)
df = df[df["unidades_vendidas"] >= 100].reset_index(drop=True)
registro.append(f"T6: se filtraron ventas con unidades_vendidas >= 100 (de {filas_antes} a {len(df)} filas).")
print(f"Filas antes: {filas_antes} | después: {len(df)}")
df.head()

Filas antes: 100 | después: 60


,fecha,tienda,ubicacion,categoria_producto,unidades_vendidas,ingresos,grupo_edad_cliente,nivel_ingreso_cliente,etiqueta_desempeno,precio_unitario,mes
0,2023-01-01,Green Market,Springfield,Electronics,150,45000,30-45,High,High,300.0,2023-01
1,2023-01-02,City Mall,Springfield,Home & Garden,200,50000,25-35,Medium,Medium,250.0,2023-01
2,2023-01-03,Book Haven,Lakeview,Books,120,18000,18-24,Low,Low,150.0,2023-01
3,2023-01-04,FashionFiesta,River City,Apparel,250,75000,40-50,High,High,300.0,2023-01
4,2023-01-05,Food World,Springfield,Food & Beverages,300,60000,20-30,Medium,Medium,200.0,2023-01


## 3. Resultado de las transformaciones (antes y después)

In [23]:
comparacion = pd.DataFrame({
    "original": {"filas": df_original.shape[0], "columnas": df_original.shape[1], "nulos": int(df_original.isna().sum().sum())},
    "transformado": {"filas": df.shape[0], "columnas": df.shape[1], "nulos": int(df.isna().sum().sum())},
})
print(comparacion)
print()
print("\n".join(registro))
df.dtypes

          original  transformado
filas          100            60
columnas        10            11
nulos            0             0

T1: se renombraron las 10 columnas a español en formato snake_case.
T2: la columna fecha se convirtió de texto a datetime.
T3: se normalizaron espacios en las 6 columnas de texto.
T4: se crearon precio_unitario (ingresos / unidades_vendidas) y mes (AAAA-MM).
T5: se eliminó id_registro (consecutivo sin valor analítico).
T6: se filtraron ventas con unidades_vendidas >= 100 (de 100 a 60 filas).


,0
fecha,datetime64[ns]
tienda,string[python]
ubicacion,string[python]
categoria_producto,string[python]
unidades_vendidas,int64
ingresos,int64
grupo_edad_cliente,string[python]
nivel_ingreso_cliente,string[python]
etiqueta_desempeno,string[python]
precio_unitario,float64


## 4. Generar el resumen `.txt` con la segunda clase
Se le pasa a la clase el **DataFrame transformado**, así el archivo corresponde a los datos finales de esta ejecución.

In [24]:
resumidor = DatasetSummarizer(
    df,
    titulo="Resumen del DataFrame transformado - EA1",
    origen=cargador.describe_source(),
    notas=registro,
)
ruta_txt = resumidor.guardar("EA1/resultados/resumen_dataframe_transformado.txt")
print("Archivo generado en:", ruta_txt)

Archivo generado en: EA1/resultados/resumen_dataframe_transformado.txt


## 5. Verificar que el resumen coincide con los datos

In [25]:
texto = ruta_txt.read_text(encoding="utf-8")
assert f"Filas: {df.shape[0]}" in texto and f"Columnas: {df.shape[1]}" in texto
print("El resumen coincide con el DataFrame transformado.\n")
print(texto)

El resumen coincide con el DataFrame transformado.

RESUMEN DEL DATAFRAME TRANSFORMADO - EA1
Generado: 2026-11-26 00:16:45
dataset_id: infinite-dataset-hub/RetailSalesPerformance
url: https://huggingface.co/datasets/infinite-dataset-hub/RetailSalesPerformance
split: train
config_name: None
revision: None

Filas: 60
Columnas: 11

Columnas, tipos de datos y valores nulos:
                                 tipo  nulos
fecha                  datetime64[ns]      0
tienda                         string      0
ubicacion                      string      0
categoria_producto             string      0
unidades_vendidas               int64      0
ingresos                        int64      0
grupo_edad_cliente             string      0
nivel_ingreso_cliente          string      0
etiqueta_desempeno             string      0
precio_unitario               float64      0
mes                            object      0
Total de valores nulos: 0

Estadísticas descriptivas (variables numéricas):
           

## 6. Qué se hizo
1. Se cargó el dataset desde Hugging Face con la clase `HFDatasetLoader` (100 filas y 10 columnas).
2. Se aplicaron seis transformaciones: renombrar columnas, convertir la fecha, normalizar texto, crear columnas calculadas, seleccionar columnas y filtrar filas.
3. El DataFrame transformado se entregó a `DatasetSummarizer`, que generó el archivo `.txt` con filas, columnas, tipos, nulos y estadísticas.

El dataset es sintético (generado con IA según su ficha), por lo que sirve para practicar el flujo y no representa un negocio real.

In [26]:
# Solo en Google Colab: descarga el resumen para subirlo al repositorio
try:
    from google.colab import files
    files.download(str(ruta_txt))
except ImportError:
    print("Fuera de Colab: el archivo está en", ruta_txt)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>